# McGill Course Advising Agent
Single-agent system using the OpenAI Agents SDK: typed tools, structured output, an input guardrail, an output guardrail, and an eval suite.

**Run order:** Run all cells top to bottom. Steps 1-9 need no API key to *define*, and Step 9 (offline tests) needs no API key or network at all. Steps 8, 10 and 11 call the OpenAI API.

## Step 1 — Install dependencies

In [1]:
!pip install -q -U openai-agents

## Step 2 — Set your OpenAI API key
The key is read from the `OPENAI_API_KEY` environment variable, or you are prompted for it. **Never paste a key into this notebook** — it ends up in the saved file and in git history.

In [2]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

OpenAI API key: ··········


## Step 3 — Fixtures (fake McGill world)

In [3]:
import json
import re
from typing import Literal, Optional

MODEL = "gpt-4.1-mini"   # change if your account does not have this model

COURSE_CATALOG = {
    "COMP 202": {"name": "Foundations of Programming",          "credits": 3, "level": 200, "description": "Intro to Python programming.",           "keywords": ["python", "intro"]},
    "COMP 250": {"name": "Introduction to Computer Science",    "credits": 3, "level": 200, "description": "Algorithms, data structures, Java.",      "keywords": ["java"]},
    "COMP 251": {"name": "Algorithms and Data Structures",      "credits": 3, "level": 200, "description": "Algorithm design and complexity.",        "keywords": ["algorithms"]},
    "COMP 302": {"name": "Programming Languages and Paradigms", "credits": 3, "level": 300, "description": "Functional and logic programming.",       "keywords": ["functional"]},
    "COMP 303": {"name": "Software Design",                     "credits": 3, "level": 300, "description": "Object-oriented design patterns.",        "keywords": ["software", "oop"]},
    "COMP 310": {"name": "Operating Systems",                   "credits": 3, "level": 300, "description": "Processes, memory, file systems.",        "keywords": ["os", "systems"]},
    "COMP 330": {"name": "Theory of Computation",               "credits": 3, "level": 300, "description": "Automata, languages, computability.",     "keywords": ["theory"]},
    "COMP 360": {"name": "Algorithm Design",                    "credits": 3, "level": 300, "description": "Advanced algorithms, NP-completeness.",   "keywords": ["algorithms"]},
    "COMP 409": {"name": "Concurrent Programming",              "credits": 3, "level": 400, "description": "Threads, synchronization, parallelism.",  "keywords": ["concurrency", "threads"]},
    "COMP 424": {"name": "Artificial Intelligence",             "credits": 3, "level": 400, "description": "Search, planning, ML basics.",            "keywords": ["ai", "ml", "machine", "learning"]},
}

PREREQUISITE_TABLE = {
    "COMP 250": ["COMP 202"],
    "COMP 251": ["COMP 250"],
    "COMP 302": ["COMP 250"],
    "COMP 303": ["COMP 250"],
    "COMP 310": ["COMP 251"],
    "COMP 330": ["COMP 251"],
    "COMP 360": ["COMP 251", "COMP 330"],
    "COMP 409": ["COMP 310"],
    "COMP 424": ["COMP 251", "COMP 302"],
}

STUDENT_PROFILE = {
    "id": "260100001",
    "name": "Alec Martin",
    "program": "B.Sc. Computer Science",
    "year": 2,
    "gpa": 3.1,
    "completed_courses": ["COMP 202", "COMP 250"],
    "current_credits": 12,
}

# Single source of truth for all policy numbers.
POLICY_RULES = {
    "max_credits_per_semester": 18,
    "advisor_approval_threshold_credits": 17,   # total credits >= this need advisor approval
    "min_gpa_for_400_level": 3.0,
    "max_concurrent_400_level": 1,
}

print("Fixtures loaded:", len(COURSE_CATALOG), "courses,", len(PREREQUISITE_TABLE), "prereq rules")

Fixtures loaded: 10 courses, 9 prereq rules


## Step 4 — Structured output types

In [4]:
from pydantic import BaseModel

# Enforced by the output schema, so the model cannot invent new labels
# (an earlier version used a free-form string and evals broke on
# 'missing_prerequisites' vs 'missing_prereq').
RiskType = Literal[
    "missing_prereq",
    "credit_overload",
    "gpa_too_low",
    "policy_flag",
    "not_in_catalog",
    "already_completed",
]

class CourseRisk(BaseModel):
    course_code: str
    risk_type: RiskType
    description: str
    blocking: bool       # True = must NOT recommend this course

class CourseRecommendation(BaseModel):
    student_id: str
    recommended_courses: list[str]
    risks: list[CourseRisk]
    advisor_approval_required: bool
    advisor_approval_reason: Optional[str]
    summary: str

print("Output types defined.")

Output types defined.


## Step 5 — Tools
The logic lives in plain `*_impl` functions (testable with no API). The `@function_tool` wrappers are thin and just serialise to JSON for the model.

In [5]:
from agents import function_tool

# ---------- helpers ----------

CODE_PATTERN = re.compile(r"\b([A-Za-z]{4})\s*-?\s*(\d{3})\b")
STOPWORDS = {"a", "an", "and", "any", "are", "available", "about", "can", "course", "courses",
             "for", "i", "in", "is", "me", "of", "on", "or", "take", "the", "there", "to",
             "what", "with", "my", "you", "do", "have"}

def normalize_code(code: str) -> str:
    """'comp251' / 'comp 251' / 'COMP-251' -> 'COMP 251'."""
    m = CODE_PATTERN.fullmatch(code.strip())
    return f"{m.group(1).upper()} {m.group(2)}" if m else code.strip().upper()

def credits_of(codes) -> int:
    return sum(COURSE_CATALOG[c]["credits"] for c in codes if c in COURSE_CATALOG)

# ---------- pure logic ----------

def search_courses_impl(query: str) -> dict:
    codes = list(dict.fromkeys(f"{a.upper()} {b}" for a, b in CODE_PATTERN.findall(query)))
    if codes:                                   # explicit course code(s): exact lookup
        matches = [c for c in codes if c in COURSE_CATALOG]
    else:                                       # keyword search, token based
        tokens = [t for t in re.findall(r"[a-z0-9]+", query.lower()) if t not in STOPWORDS]
        if not tokens:                          # "what's available?" -> browse everything
            matches = list(COURSE_CATALOG)
        else:
            scored = []
            for code, info in COURSE_CATALOG.items():
                hay = " ".join([code, info["name"], info["description"], *info["keywords"]]).lower()
                hay_tokens = set(re.findall(r"[a-z0-9]+", hay))
                score = sum(t in hay_tokens for t in tokens)
                if score:
                    scored.append((code, score))
            matches = [c for c, _ in sorted(scored, key=lambda s: -s[1])]
    if not matches:
        return {"results": [], "message": f"No courses found matching '{query}'."}
    return {"results": [{
        "code": c, "name": COURSE_CATALOG[c]["name"], "credits": COURSE_CATALOG[c]["credits"],
        "description": COURSE_CATALOG[c]["description"], "level": COURSE_CATALOG[c]["level"],
        "prerequisites": PREREQUISITE_TABLE.get(c, []),
    } for c in matches]}


def check_prerequisites_impl(student_id: str, course_code: str, profile=STUDENT_PROFILE) -> dict:
    course_code = normalize_code(course_code)
    if student_id != profile["id"]:
        return {"eligible": False, "course": course_code, "risk_type": "policy_flag", "reason": "Student not found."}
    if course_code not in COURSE_CATALOG:
        return {"eligible": False, "course": course_code, "risk_type": "not_in_catalog",
                "reason": f"Course {course_code} does not exist in the catalog."}
    completed = set(profile["completed_courses"])
    if course_code in completed:
        return {"eligible": False, "course": course_code, "risk_type": "already_completed",
                "reason": f"Student has already completed {course_code}."}
    required = PREREQUISITE_TABLE.get(course_code, [])
    missing = [c for c in required if c not in completed]
    if missing:
        return {"eligible": False, "course": course_code, "risk_type": "missing_prereq",
                "missing_prerequisites": missing, "reason": f"Student has not completed: {', '.join(missing)}"}
    return {"eligible": True, "course": course_code, "prerequisites_met": required,
            "reason": "All prerequisites satisfied."}


def flag_policy_risk_impl(student_id: str, proposed_course_codes: list, profile=STUDENT_PROFILE,
                          rules=POLICY_RULES) -> dict:
    if student_id != profile["id"]:
        return {"error": "Student not found."}
    codes = list(dict.fromkeys(normalize_code(c) for c in proposed_course_codes))
    completed = set(profile["completed_courses"])
    flags, reasons = [], []

    for c in codes:
        if c not in COURSE_CATALOG:
            flags.append({"course_code": c, "rule": "NOT_IN_CATALOG", "risk_type": "not_in_catalog",
                          "blocking": True, "description": f"{c} is not in the catalog."})
        elif c in completed:
            flags.append({"course_code": c, "rule": "ALREADY_COMPLETED", "risk_type": "already_completed",
                          "blocking": True, "description": f"{c} was already completed."})

    valid = [c for c in codes if c in COURSE_CATALOG and c not in completed]
    proposed_credits = credits_of(valid)
    total = profile["current_credits"] + proposed_credits
    cap, threshold = rules["max_credits_per_semester"], rules["advisor_approval_threshold_credits"]

    if total > cap:
        flags.append({"rule": "CREDIT_OVERLOAD_BLOCK", "risk_type": "credit_overload", "blocking": True,
                      "description": f"Total would be {total} credits, exceeding the {cap}-credit maximum."})
    elif total >= threshold:
        reasons.append(f"Registering for {total} credits meets the {threshold}-credit advisor-approval threshold.")

    level_400 = [c for c in valid if COURSE_CATALOG[c]["level"] == 400]
    if level_400 and profile["gpa"] < rules["min_gpa_for_400_level"]:
        flags.append({"rule": "GPA_BELOW_400_LEVEL_MINIMUM", "risk_type": "gpa_too_low", "blocking": False,
                      "description": f"GPA {profile['gpa']} is below the {rules['min_gpa_for_400_level']} minimum for {', '.join(level_400)}."})
        reasons.append("GPA is below the 400-level minimum.")
    if len(level_400) > rules["max_concurrent_400_level"]:
        flags.append({"rule": "TOO_MANY_400_LEVEL_COURSES", "risk_type": "policy_flag", "blocking": False,
                      "description": f"Proposing {len(level_400)} 400-level courses; max recommended is {rules['max_concurrent_400_level']}."})

    return {"current_credits": profile["current_credits"], "proposed_credits": proposed_credits,
            "total_credits": total, "max_allowed": cap,
            "advisor_approval_required": bool(reasons),
            "advisor_approval_reason": " ".join(reasons) or None,
            "policy_flags": flags}

# ---------- thin tool wrappers ----------

@function_tool
def search_courses(query: str) -> str:
    """Search the McGill course catalog by course code or keyword. Returns matching courses with code, name, credits, description, level and prerequisites."""
    return json.dumps(search_courses_impl(query))

@function_tool
def check_prerequisites(student_id: str, course_code: str) -> str:
    """Check whether the student can take a course: it exists, is not already completed, and all prerequisites are done."""
    return json.dumps(check_prerequisites_impl(student_id, course_code))

@function_tool
def flag_policy_risk(student_id: str, proposed_course_codes: list[str]) -> str:
    """Evaluate a proposed course list against registration policies (credit cap, advisor-approval threshold, GPA and 400-level rules)."""
    return json.dumps(flag_policy_risk_impl(student_id, proposed_course_codes))

print("Tools defined: search_courses, check_prerequisites, flag_policy_risk")

Tools defined: search_courses, check_prerequisites, flag_policy_risk


## Step 6 — Guardrails
- **`CREDIT_OVERLOAD_BLOCK` (input):** stops requests that *explicitly* ask for more than the credit cap, before the agent runs.
- **`RECOMMENDATION_VALIDATOR` (output):** re-checks the final `CourseRecommendation` in plain code (catalog, prerequisites, credit cap, approval flag), regardless of what the model said or how the request was phrased.

In [6]:
from agents import (GuardrailFunctionOutput, InputGuardrail, OutputGuardrail,
                    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered)

CREDIT_MENTION = re.compile(r"\b(\d{1,3})[\s-]*credits?\b", re.IGNORECASE)

def find_credit_overload(text: str) -> Optional[int]:
    """Return the first explicit credit figure above the cap, else None."""
    for m in CREDIT_MENTION.findall(text):
        if int(m) > POLICY_RULES["max_credits_per_semester"]:
            return int(m)
    return None

async def credit_overload_guardrail(ctx, agent, input_data) -> GuardrailFunctionOutput:
    text = input_data if isinstance(input_data, str) else str(input_data)
    n = find_credit_overload(text)
    if n is not None:
        return GuardrailFunctionOutput(
            output_info=f"CREDIT_OVERLOAD_BLOCK: {n} credits exceeds the {POLICY_RULES['max_credits_per_semester']}-credit semester maximum.",
            tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info="ok", tripwire_triggered=False)


def validate_recommendation(rec: CourseRecommendation, profile=STUDENT_PROFILE) -> list:
    """Deterministic post-check. Returns a list of problems (empty list = OK)."""
    problems = []
    completed = set(profile["completed_courses"])
    recommended = [normalize_code(c) for c in rec.recommended_courses]
    for c in recommended:
        if c not in COURSE_CATALOG:
            problems.append(f"{c} is not in the catalog")
        elif c in completed:
            problems.append(f"{c} was already completed")
        else:
            missing = [p for p in PREREQUISITE_TABLE.get(c, []) if p not in completed]
            if missing:
                problems.append(f"{c} has unmet prerequisites: {', '.join(missing)}")
    policy = flag_policy_risk_impl(profile["id"], recommended, profile)
    if policy["total_credits"] > policy["max_allowed"]:
        problems.append(f"recommendation totals {policy['total_credits']} credits (max {policy['max_allowed']})")
    if policy["advisor_approval_required"] and not rec.advisor_approval_required:
        problems.append("advisor approval is required by policy but was not flagged")
    return problems

async def recommendation_validator(ctx, agent, output) -> GuardrailFunctionOutput:
    problems = validate_recommendation(output)
    return GuardrailFunctionOutput(output_info=problems or "ok", tripwire_triggered=bool(problems))

print("Guardrails defined: CREDIT_OVERLOAD_BLOCK (input), RECOMMENDATION_VALIDATOR (output)")

Guardrails defined: CREDIT_OVERLOAD_BLOCK (input), RECOMMENDATION_VALIDATOR (output)


## Step 7 — Build the agent

In [7]:
from agents import Agent

SYSTEM_PROMPT = f"""
You are McGill Course Advisor, a single-purpose AI agent that helps undergraduate
students plan their course registration.

Tools:
- search_courses: find courses in the catalog by code or keyword
- check_prerequisites: verify a student can take a course (also reports unknown and already-completed courses)
- flag_policy_risk: check credit limits and policy rules for a list of courses

Student context:
{json.dumps(STUDENT_PROFILE, indent=2)}

Procedure for every request:
1. Work out which courses the student is asking about. If they name a topic rather than a
   course code, or ask what is available, call search_courses.
2. Call check_prerequisites for EVERY candidate course.
3. Build a proposed list from the eligible courses and call flag_policy_risk on it.
4. If flag_policy_risk reports a blocking flag, remove courses (keep the ones the student
   listed first) until no blocking flag remains, then call flag_policy_risk again on the
   reduced list.
5. Return a CourseRecommendation.

Rules:
- recommended_courses contains ONLY courses that passed check_prerequisites and have no
  blocking flag in the final flag_policy_risk call.
- Every course you do not recommend gets a CourseRisk with blocking=True. Use the risk_type
  the tools report (missing_prereq, not_in_catalog, already_completed, credit_overload).
- If you dropped courses to stay under the credit cap, add a credit_overload risk for each.
- Set advisor_approval_required and advisor_approval_reason from the FINAL flag_policy_risk
  result for the recommended list.
- Use the student id from the student context for student_id.
- Do not invent courses, prerequisites or policies. If a course is not in the catalog, say
  so clearly in the summary.
- Keep the summary to 1-3 sentences.
"""

advisor_agent = Agent(
    name="McGillCourseAdvisor",
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=[search_courses, check_prerequisites, flag_policy_risk],
    output_type=CourseRecommendation,
    input_guardrails=[
        # run_in_parallel=False: the guardrail finishes BEFORE the agent starts,
        # so no tool is ever called for a blocked request.
        InputGuardrail(guardrail_function=credit_overload_guardrail,
                       name="CREDIT_OVERLOAD_BLOCK", run_in_parallel=False),
    ],
    output_guardrails=[
        OutputGuardrail(guardrail_function=recommendation_validator,
                        name="RECOMMENDATION_VALIDATOR"),
    ],
)

print("Agent created:", advisor_agent.name)

Agent created: McGillCourseAdvisor


## Step 8 — Run a single query

In [8]:
from agents import Runner

async def run_case(query: str) -> dict:
    """Run the agent once. Returns {'kind': 'output' | 'input_guardrail' | 'output_guardrail', ...}.
    Any other exception (network, auth, SDK bug) is NOT swallowed - it propagates."""
    try:
        result = await Runner.run(advisor_agent, query)
        return {"kind": "output", "output": result.final_output}
    except InputGuardrailTripwireTriggered as e:
        return {"kind": "input_guardrail", "info": e.guardrail_result.output.output_info}
    except OutputGuardrailTripwireTriggered as e:
        return {"kind": "output_guardrail", "info": e.guardrail_result.output.output_info}

async def run_query(query: str):
    print(f"\nQUERY: {query}")
    print("-" * 50)
    r = await run_case(query)
    if r["kind"] != "output":
        print(f"BLOCKED by {r['kind']}: {r['info']}")
        return r
    output = r["output"]
    print(f"Recommended : {output.recommended_courses}")
    print(f"Needs approval: {output.advisor_approval_required}")
    if output.advisor_approval_reason:
        print(f"Approval reason: {output.advisor_approval_reason}")
    if output.risks:
        print("Risks:")
        for risk in output.risks:
            print(f"  [{risk.risk_type}] {risk.course_code}: {risk.description} (blocking={risk.blocking})")
    print(f"Summary: {output.summary}")
    return output

await run_query("Can I take COMP 251 this semester?")


QUERY: Can I take COMP 251 this semester?
--------------------------------------------------
Recommended : ['COMP 251']
Needs approval: False
Summary: You are eligible to take COMP 251 this semester as you have met all the prerequisites and adding this course will keep you within the credit limit.


CourseRecommendation(student_id='260100001', recommended_courses=['COMP 251'], risks=[], advisor_approval_required=False, advisor_approval_reason=None, summary='You are eligible to take COMP 251 this semester as you have met all the prerequisites and adding this course will keep you within the credit limit.')

## Step 9 — Offline tests (no API key, no network)
Deterministic checks of the tools, the input guardrail's regex and the output validator. These should **always** pass; if one fails it is a code bug, not model variance.

In [9]:
def test_offline():
    P = STUDENT_PROFILE["id"]

    # --- search ---
    assert [r["code"] for r in search_courses_impl("COMP 251")["results"]] == ["COMP 251"]
    assert [r["code"] for r in search_courses_impl("comp251")["results"]] == ["COMP 251"]
    assert "COMP 424" in [r["code"] for r in search_courses_impl("AI")["results"]]
    assert "COMP 424" in [r["code"] for r in search_courses_impl("AI or machine learning courses")["results"]]
    assert "COMP 303" in [r["code"] for r in search_courses_impl("software design")["results"]]
    assert search_courses_impl("COMP 999")["results"] == []
    assert search_courses_impl("basket weaving")["results"] == []
    assert len(search_courses_impl("what is available?")["results"]) == len(COURSE_CATALOG)

    # --- prerequisites ---
    assert check_prerequisites_impl(P, "COMP 251")["eligible"] is True
    r = check_prerequisites_impl(P, "COMP 360")
    assert r["eligible"] is False and r["risk_type"] == "missing_prereq"
    assert r["missing_prerequisites"] == ["COMP 251", "COMP 330"]
    assert check_prerequisites_impl(P, "COMP 999")["risk_type"] == "not_in_catalog"
    assert check_prerequisites_impl(P, "COMP 250")["risk_type"] == "already_completed"
    assert check_prerequisites_impl(P, "comp251")["eligible"] is True
    assert check_prerequisites_impl("bad-id", "COMP 251")["eligible"] is False

    # --- policy: credits (student is at 12) ---
    p = flag_policy_risk_impl(P, ["COMP 251"])
    assert p["total_credits"] == 15 and p["advisor_approval_required"] is False and p["policy_flags"] == []
    p = flag_policy_risk_impl(P, ["COMP 251", "COMP 302"])
    assert p["total_credits"] == 18 and p["advisor_approval_required"] is True
    assert not any(f["blocking"] for f in p["policy_flags"])
    p = flag_policy_risk_impl(P, ["COMP 251", "COMP 302", "COMP 303"])
    assert p["total_credits"] == 21
    assert any(f["risk_type"] == "credit_overload" and f["blocking"] for f in p["policy_flags"])
    assert flag_policy_risk_impl(P, ["COMP 251", "COMP 251"])["proposed_credits"] == 3   # de-duplicated
    p = flag_policy_risk_impl(P, ["COMP 999", "COMP 250"])
    assert {f["risk_type"] for f in p["policy_flags"]} == {"not_in_catalog", "already_completed"}

    # --- policy: GPA / 400-level (uses a low-GPA copy of the profile) ---
    low = {**STUDENT_PROFILE, "gpa": 2.7}
    p = flag_policy_risk_impl(P, ["COMP 424"], profile=low)
    assert any(f["risk_type"] == "gpa_too_low" for f in p["policy_flags"])
    assert p["advisor_approval_required"] is True
    p = flag_policy_risk_impl(P, ["COMP 424"])            # real student has gpa 3.1
    assert not any(f["risk_type"] == "gpa_too_low" for f in p["policy_flags"])
    p = flag_policy_risk_impl(P, ["COMP 409", "COMP 424"])
    assert any(f["rule"] == "TOO_MANY_400_LEVEL_COURSES" for f in p["policy_flags"])

    # --- input guardrail regex ---
    assert find_credit_overload("Please register me for 25 credits") == 25
    assert find_credit_overload("that would bring me to 21 credits total") == 21
    assert find_credit_overload("a 30-credit load") == 30
    assert find_credit_overload("Can I take COMP 251?") is None
    assert find_credit_overload("I have 12 credits already") is None

    # --- output validator ---
    def rec(courses, approval=False):
        return CourseRecommendation(student_id=P, recommended_courses=courses, risks=[],
                                    advisor_approval_required=approval, advisor_approval_reason=None, summary="")
    assert validate_recommendation(rec(["COMP 251"])) == []
    assert validate_recommendation(rec([])) == []
    assert validate_recommendation(rec(["COMP 251", "COMP 302"], approval=True)) == []
    assert any("not in the catalog" in x for x in validate_recommendation(rec(["COMP 999"])))
    assert any("unmet prerequisites" in x for x in validate_recommendation(rec(["COMP 360"])))
    assert any("already completed" in x for x in validate_recommendation(rec(["COMP 250"])))
    assert any("max 18" in x for x in validate_recommendation(rec(["COMP 251", "COMP 302", "COMP 303"], approval=True)))
    assert any("not flagged" in x for x in validate_recommendation(rec(["COMP 251", "COMP 302"], approval=False)))

    print("All offline tests passed.")

test_offline()

All offline tests passed.


## Step 10 — Live eval cases (calls the API)
Each case runs once against the real model, so results can vary between runs. Cases 05 and 07 also exercise the guardrails.

In [10]:
CAP = POLICY_RULES["max_credits_per_semester"]

EVAL_CASES = [
    {"id": "eval_01", "description": "Happy path: eligible course",
     "query": "Can I take COMP 251 this semester?",
     "expect_recommended": ["COMP 251"], "expect_approval_required": False},

    {"id": "eval_02", "description": "Happy path: topic search finds an eligible course",
     "query": "Is there a software design course I can take?",
     "expect_recommended_contains_any": ["COMP 303"]},

    {"id": "eval_03", "description": "Happy path: two valid courses land exactly on 18 credits -> approval",
     "query": "I want to take COMP 251 and COMP 302 this semester.",
     "expect_recommended": ["COMP 251", "COMP 302"], "expect_approval_required": True},

    {"id": "eval_04_edge", "description": "EDGE: missing prerequisites for COMP 360",
     "query": "I want to register for COMP 360.",
     "expect_recommended": [], "expect_risk": ("missing_prereq", "COMP 360")},

    {"id": "eval_05_edge", "description": "EDGE: input guardrail fires on explicit credit overload",
     "query": "Please register me for 25 credits this semester.",
     "expect_blocked_by": "input_guardrail"},

    {"id": "eval_06_edge", "description": "EDGE: course does not exist",
     "query": "Can I take COMP 999?",
     "expect_recommended": [], "expect_risk": ("not_in_catalog", "COMP 999")},

    {"id": "eval_07_edge", "description": "EDGE: overload with no credit number stated (input guardrail cannot see it)",
     "query": "I want to add COMP 251, COMP 302, and COMP 303.",
     "expect_within_credit_cap": True, "expect_risk": ("credit_overload", None)},

    {"id": "eval_08_edge", "description": "EDGE: AI courses exist but student lacks prerequisites",
     "query": "What AI or machine learning courses are available for me?",
     "expect_not_recommended": ["COMP 424"], "expect_risk": ("missing_prereq", "COMP 424")},
]

def check_case(case: dict, r: dict) -> list:
    """Return a list of failure messages (empty = pass)."""
    fails = []
    if "expect_blocked_by" in case:
        if r["kind"] != case["expect_blocked_by"]:
            fails.append(f"expected {case['expect_blocked_by']}, got {r['kind']}")
        return fails

    if r["kind"] == "output_guardrail" and case.get("expect_within_credit_cap"):
        return []        # model overshot the cap, but the output guardrail caught it - acceptable
    if r["kind"] != "output":
        return [f"unexpected block: {r['kind']} - {r['info']}"]

    out = r["output"]
    rec = set(out.recommended_courses)
    if "expect_recommended" in case and set(case["expect_recommended"]) != rec:
        fails.append(f"recommended {sorted(rec)}, expected {case['expect_recommended']}")
    if "expect_recommended_contains_any" in case and not rec & set(case["expect_recommended_contains_any"]):
        fails.append(f"expected one of {case['expect_recommended_contains_any']} in {sorted(rec)}")
    if "expect_not_recommended" in case and rec & set(case["expect_not_recommended"]):
        fails.append(f"should not recommend {sorted(rec & set(case['expect_not_recommended']))}")
    if "expect_approval_required" in case and out.advisor_approval_required != case["expect_approval_required"]:
        fails.append(f"approval_required={out.advisor_approval_required}, expected {case['expect_approval_required']}")
    if "expect_risk" in case:
        rtype, rcourse = case["expect_risk"]
        if not any(x.risk_type == rtype and (rcourse is None or normalize_code(x.course_code) == rcourse) for x in out.risks):
            fails.append(f"expected risk {rtype}/{rcourse} in {[(x.risk_type, x.course_code) for x in out.risks]}")
    if case.get("expect_within_credit_cap"):
        total = STUDENT_PROFILE["current_credits"] + credits_of(out.recommended_courses)
        if total > CAP:
            fails.append(f"recommendation totals {total} credits (max {CAP})")
    return fails

async def run_evals():
    passed = 0
    print("=" * 60); print("LIVE EVAL RUN"); print("=" * 60)
    for case in EVAL_CASES:
        print(f"\n[{case['id']}] {case['description']}")
        try:
            r = await run_case(case["query"])
            fails = check_case(case, r)
        except Exception as e:                      # real errors count as failures, never as passes
            fails = [f"ERROR {type(e).__name__}: {e}"]
        if fails:
            for f in fails:
                print(f"  FAIL - {f}")
        else:
            print("  PASS")
            passed += 1
    print(f"\n{'=' * 60}\nRESULTS: {passed} passed, {len(EVAL_CASES) - passed} failed out of {len(EVAL_CASES)} cases\n{'=' * 60}")

await run_evals()

LIVE EVAL RUN

[eval_01] Happy path: eligible course
  PASS

[eval_02] Happy path: topic search finds an eligible course
  PASS

[eval_03] Happy path: two valid courses land exactly on 18 credits -> approval
  PASS

[eval_04_edge] EDGE: missing prerequisites for COMP 360
  PASS

[eval_05_edge] EDGE: input guardrail fires on explicit credit overload
  PASS

[eval_06_edge] EDGE: course does not exist
  PASS

[eval_07_edge] EDGE: overload with no credit number stated (input guardrail cannot see it)
  PASS

[eval_08_edge] EDGE: AI courses exist but student lacks prerequisites
  PASS

RESULTS: 8 passed, 0 failed out of 8 cases


## Step 11 — Try any query

In [11]:
# Change this to whatever you want to ask the agent
await run_query("What courses can I take next semester?")


QUERY: What courses can I take next semester?
--------------------------------------------------
BLOCKED by output_guardrail: ['COMP 204 is not in the catalog', 'COMP 206 is not in the catalog', 'MATH 240 is not in the catalog', 'MATH 241 is not in the catalog']


{'kind': 'output_guardrail',
 'info': ['COMP 204 is not in the catalog',
  'COMP 206 is not in the catalog',
  'MATH 240 is not in the catalog',
  'MATH 241 is not in the catalog']}